# Specimen 04 — RAG Pipeline

Goal: combine retrieval + generation, and evaluate the retrieval quality itself — not just whether the final answer sounds right.

In [1]:
import os
from dotenv import load_dotenv
import anthropic
from sentence_transformers import SentenceTransformer
import numpy as np

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
embedder = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 1. Reuse specimen 03's chunk store

Same chunks, same embedding function and retrieval logic.

In [2]:
MODEL = 'claude-opus-5'

documents = [
    "The Wright brothers achieved the first powered flight in 1903 near Kitty Hawk, North Carolina.",
    "Python is a high-level programming language known for its readability and broad standard library.",
    "The mitochondria is the organelle responsible for producing ATP, the energy currency of the cell.",
    "Photosynthesis converts light energy into chemical energy stored in glucose.",
    "The stock market crash of 1929 triggered the Great Depression in the United States.",
    "Neural networks are loosely inspired by the structure of biological neurons in the brain.",
    "The Great Wall of China was built over centuries to protect against invasions from the north.",
    "Machine learning models improve their performance by learning patterns from training data.",
    "The French Revolution began in 1789 and led to the end of the monarchy in France.",
    "DNA carries the genetic instructions used in the growth and functioning of living organisms.",
    "Climate change is driven largely by the accumulation of greenhouse gases in the atmosphere.",
    "The Roman Empire at its height stretched from Britain to the Middle East.",
    "Quantum computers use qubits, which can exist in superposition, unlike classical bits.",
    "The human heart pumps blood through a network of arteries, veins, and capillaries.",
    "Shakespeare wrote 37 plays and over 150 sonnets during the late 16th and early 17th centuries.",
    "Renewable energy sources like solar and wind are becoming cheaper than fossil fuels in many regions.",
    "The Amazon rainforest produces roughly 20% of the world's oxygen and is called Earth's lungs.",
    "Blockchain is a distributed ledger technology that underlies cryptocurrencies like Bitcoin.",
    "Volcanic eruptions occur when magma, gases, and ash escape from below Earth's crust.",
    "The Apollo 11 mission landed the first humans on the Moon in July 1969.",
    "Antibiotics work by killing bacteria or stopping their growth, but are ineffective against viruses.",
    "The Industrial Revolution transformed manufacturing from hand production to machines.",
    "Coral reefs support roughly 25% of all marine species despite covering less than 1% of the ocean floor.",
    "GDP measures the total monetary value of goods and services produced within a country.",
    "The printing press, invented by Gutenberg around 1440, revolutionized the spread of information.",
]

doc_embeddings = embedder.encode(documents, convert_to_numpy=True)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query, top_k=3):
    q_vec = embedder.encode([query], convert_to_numpy=True)[0]
    scores = [cosine_similarity(q_vec, doc_vec) for doc_vec in doc_embeddings]
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

print(f'{len(documents)} chunks ready, retrieval reused from specimen 03')

25 chunks ready, retrieval reused from specimen 03


## 2. Build the RAG loop

Given a question: retrieve top-k chunks, stuff them into the prompt as context, call the LLM to answer using only that context.

In [3]:
def rag_answer(question, top_k=3, max_tokens=400):
    retrieved = retrieve(question, top_k=top_k)
    context = '\n\n'.join(f'[{i+1}] {doc}' for i, (doc, score) in enumerate(retrieved))
    prompt = (
        f"Answer the question using ONLY the numbered context below. "
        f"If the context does not contain the answer, say you don't know — do not use outside knowledge.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    response = client.messages.create(
        model=MODEL,
        thinking={"type": "disabled"},
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = ''.join(b.text for b in response.content if b.type == 'text')
    return answer, retrieved

answer, retrieved = rag_answer("When did humans first land on the Moon?")
print(answer)
print('\nRetrieved chunks:')
for doc, score in retrieved:
    print(f'  {score:.3f}  {doc}')

July 1969 — according to [1], the Apollo 11 mission landed the first humans on the Moon then.

Retrieved chunks:
  0.757  The Apollo 11 mission landed the first humans on the Moon in July 1969.
  0.340  The Wright brothers achieved the first powered flight in 1903 near Kitty Hawk, North Carolina.
  0.112  Volcanic eruptions occur when magma, gases, and ash escape from below Earth's crust.


## 3. Force the model to cite

Instruct it to reference which chunk(s) it used, so you can check whether the answer is actually grounded in what was retrieved.

In [4]:
def rag_answer_with_citations(question, top_k=3, max_tokens=400):
    retrieved = retrieve(question, top_k=top_k)
    context = '\n\n'.join(f'[{i+1}] {doc}' for i, (doc, score) in enumerate(retrieved))
    prompt = (
        f"Answer the question using ONLY the numbered context below. "
        f"After your answer, on a new line write 'Sources: ' followed by the numbers of the context chunks you used, e.g. 'Sources: [1], [3]'. "
        f"If the context does not contain the answer, say you don't know and write 'Sources: none'.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    response = client.messages.create(
        model=MODEL,
        thinking={"type": "disabled"},
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = ''.join(b.text for b in response.content if b.type == 'text')
    return answer, retrieved

answer, retrieved = rag_answer_with_citations("When did humans first land on the Moon?")
print(answer)

Humans first landed on the Moon in July 1969, during the Apollo 11 mission.

Sources: [1]


## 4. Evaluate retrieval quality directly

For 5-10 questions where you know the right source chunk ahead of time, check whether your retriever actually returned it in the top-k. Report this separately from whether the final answer was correct.

In [5]:
eval_set = [
    ("When did humans first land on the Moon?", "The Apollo 11 mission landed the first humans on the Moon in July 1969."),
    ("What organelle produces a cell's energy?", "The mitochondria is the organelle responsible for producing ATP, the energy currency of the cell."),
    ("What caused the Great Depression?", "The stock market crash of 1929 triggered the Great Depression in the United States."),
    ("How much of the world's oxygen does the Amazon produce?", "The Amazon rainforest produces roughly 20% of the world's oxygen and is called Earth's lungs."),
    ("What percentage of marine species do coral reefs support?", "Coral reefs support roughly 25% of all marine species despite covering less than 1% of the ocean floor."),
    ("Who invented the printing press and when?", "The printing press, invented by Gutenberg around 1440, revolutionized the spread of information."),
]

hits = 0
for question, correct_chunk in eval_set:
    retrieved = retrieve(question, top_k=3)
    retrieved_docs = [doc for doc, score in retrieved]
    hit = correct_chunk in retrieved_docs
    hits += hit
    print(f'{"HIT " if hit else "MISS"}  {question}')

print(f'\nRetrieval precision@3: {hits}/{len(eval_set)} = {hits/len(eval_set):.1%}')

HIT   When did humans first land on the Moon?
HIT   What organelle produces a cell's energy?
HIT   What caused the Great Depression?


HIT   How much of the world's oxygen does the Amazon produce?
HIT   What percentage of marine species do coral reefs support?
HIT   Who invented the printing press and when?

Retrieval precision@3: 6/6 = 100.0%


## 5. Break it on purpose

Ask a question your document set can't answer. Does the model honestly say so, or hallucinate an answer anyway?

In [6]:
unanswerable_question = "What is the capital of Australia?"
answer, retrieved = rag_answer(unanswerable_question)
print('Question:', unanswerable_question)
print('Answer:', answer)
print('\nRetrieved (irrelevant) chunks:')
for doc, score in retrieved:
    print(f'  {score:.3f}  {doc}')

Question: What is the capital of Australia?
Answer: The context provided does not contain the answer to this question. The numbered passages cover the Amazon rainforest, the Roman Empire, and the Python programming language — none of them mention Australia or its capital.

I don't know, based on the given context.

Retrieved (irrelevant) chunks:
  0.174  The Amazon rainforest produces roughly 20% of the world's oxygen and is called Earth's lungs.
  0.156  The Roman Empire at its height stretched from Britain to the Middle East.
  0.124  Python is a high-level programming language known for its readability and broad standard library.


## 6. Compare against no-RAG

Same questions, no retrieved context at all. Compare answer quality side by side.

In [7]:
def no_rag_answer(question, max_tokens=400):
    response = client.messages.create(
        model=MODEL,
        thinking={"type": "disabled"},
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": question}],
    )
    return ''.join(b.text for b in response.content if b.type == 'text')

comparison_questions = [
    "How much of the world's oxygen does the Amazon produce?",
    "What is the capital of Australia?",
]

for question in comparison_questions:
    rag_ans, _ = rag_answer(question)
    plain_ans = no_rag_answer(question)
    print(f'Question: {question}')
    print(f'  RAG:     {rag_ans}')
    print(f'  No-RAG:  {plain_ans}')
    print()

Question: How much of the world's oxygen does the Amazon produce?
  RAG:     According to the context, the Amazon rainforest produces roughly 20% of the world's oxygen, and for this reason it is called "Earth's lungs" [1].
  No-RAG:  The Amazon rainforest is often described as "the lungs of the planet," producing 20% of the world's oxygen — but this figure is misleading, and scientists have pushed back on it substantially.

**The actual numbers:** The Amazon accounts for roughly 6–9% of global photosynthesis on land. But land plants only produce part of Earth's oxygen — marine phytoplankton contribute a comparable or larger share. So the Amazon's gross contribution to global oxygen production is more like 6–9% at most, and lower once ocean output is factored in.

**The bigger point:** Net oxygen contribution is close to zero. Trees produce oxygen through photosynthesis, but they also consume it through respiration. And when leaves, wood, and other organic matter decompose — which happe

Question: What is the capital of Australia?
  RAG:     The context provided does not contain the answer to your question. The three passages cover the Amazon rainforest, the Roman Empire, and the Python programming language — none of them mention Australia or its capital.
  No-RAG:  The capital of Australia is Canberra. It's located in the Australian Capital Territory (ACT), between Sydney and Melbourne. Canberra was purpose-built as the capital in the early 20th century as a compromise, since Sydney and Melbourne both wanted the honor. The city was formally established in 1913, and Parliament moved there from Melbourne in 1927.

